In [31]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, Trainer, TrainingArguments
from scipy.optimize import linear_sum_assignment

random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [32]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('alchemy-aicc-round-4')

print("Path to competition files:", path)

Path to competition files: C:\Users\raian\.cache\kagglehub\competitions\alchemy-aicc-round-4


In [33]:
endpoint = 'bert-base-uncased'
model = AutoModelForSequenceClassification.from_pretrained(endpoint, num_labels=1)
tokenizer = AutoTokenizer.from_pretrained(endpoint)

candidates_df = pd.read_csv(path + '/candidates.csv')
train_df = pd.read_csv(path + '/train.csv')
test_df = pd.read_csv(path + '/test.csv')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
candidates = list(candidates_df['result'])
train_pool = list(train_df['result'])

flipped = train_df.rename(columns={'item1': 'item2', 'item2': 'item1'})
train_aug = pd.concat([train_df, flipped]).drop_duplicates().reset_index(drop=True)

train_ds = Dataset.from_pandas(train_aug[['item1', 'item2', 'result']])

In [35]:
for param in model.parameters():
    param.requires_grad = False

In [36]:
correct = {(row['item1'], row['item2']): row['result'] for row in train_ds}

def collate(batch):
    a  = [x['item1']  for x in batch]
    b  = [x['item2']  for x in batch]
    c  = [x['result'] for x in batch]
    n  = len(batch)
    triples = []
    for i in range(n):
        j, k = random.randrange(n), random.randrange(n)
        triples += [(a[i], b[i], c[i]),      # positive
                    (a[i], b[j], c[i]),      # wrong second ingredient
                    (a[i], b[i], c[k])]      # wrong result
    texts  = [f'Combining {x} and {y} creates {z}' for x, y, z in triples]
    labels = torch.tensor([correct.get((x, y)) == z for x, y, z in triples]).long()
    enc = tokenizer(texts, padding=True, truncation=True, max_length=48, return_tensors='pt')
    enc['labels'] = labels
    return enc

In [37]:
# from peft import LoraConfig, get_peft_model, TaskType

# config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,
#     r=32,
#     target_modules=['query', 'value'],
#     modules_to_save=['classifier'],
#     lora_dropout=0.1,
# )

# model =get_peft_model(model, config)
# model.print_trainable_parameters()

In [44]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

for p in model.bert.embeddings.parameters():
    p.requires_grad = False
for layer in model.bert.encoder.layer[:7]:
    for p in layer.parameters():
        p.requires_grad = False

args = TrainingArguments(
    output_dir='out',
    per_device_train_batch_size=8,
    num_train_epochs=10,
    learning_rate=2e-5,
    warmup_steps=30,
    logging_strategy='epoch',
    remove_unused_columns=False,
)

trainer = Trainer(model=model, args=args,
                             train_dataset=train_ds, data_collator=collate)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
37,0.680825
74,0.636307
111,0.618338
148,0.565011
185,0.518031
222,0.472760
259,0.429129
296,0.418679
333,0.391643
370,0.413431


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [46]:
model.eval()
score_mat = np.zeros((len(test_df), len(candidates)))


def score(texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i + bs], padding=True, truncation=True,
                        max_length=48, return_tensors='pt').to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits
            scores = (logits[:, 1] - logits[:, 0]).cpu().numpy()
            out.append(scores)

    return np.concatenate(out)


for i, row in test_df.reset_index(drop=True).iterrows():
    fwd = score([f"Combining {row['item1']} and {row['item2']} creates {c}" for c in candidates])
    rev = score([f"Combining {row['item2']} and {row['item1']} creates {c}" for c in candidates])
    
    score_mat[i] = (fwd + rev) / 2

_, col_assignments = linear_sum_assignment(-score_mat)
predictions = [candidates[c] for c in col_assignments]

In [47]:
predictions

['satellite',
 'tequila',
 'whiskey',
 'milk',
 'concrete',
 'needle',
 'road',
 'dog',
 'armchair',
 'lasso',
 'sushi',
 'bouquet',
 'titanic',
 'fish',
 'storm',
 'calendar',
 'helicopter',
 'light',
 'rocket',
 'radiation',
 'butterfly',
 'airplane',
 'vodka',
 'moss',
 'unicorn',
 'scorpion',
 'dragon',
 'snowman',
 'house',
 'seeds',
 'vinegar',
 'panda',
 'polar bear',
 'livestock',
 'golem',
 'book',
 'carbon',
 'cement',
 'termites',
 'chocolate',
 'ghoul',
 'claws',
 'meat',
 'bulb',
 'tobacco',
 'tool',
 'soldier',
 'fisherman',
 'bomb',
 'virus',
 'spellbook',
 'spell',
 'acid',
 'necromancer',
 'potion',
 'wand',
 'tiger',
 'radar',
 'galaxy',
 'salt water',
 'sulfur',
 'fabric',
 'demon',
 'metal',
 'tower',
 'mud',
 'alien',
 'wheat',
 'caviar',
 'wax']